实现Variable类

In [ ]:
class Variable:
    def __init__(self, data):
        self.data = data

Function类的实现

In [ ]:
class Function:
    def __call__(self, input):
        x = input.data
        # y = x ** 2
        y = self.forward(x) # 集体计算方法在forward中进行
        output = Variable(y)
        return output

    def forward(self, x):
        raise NotImplementedError()

用于计算平方的类

In [ ]:
class Square(Function):
    def forward(self, x):
        return x ** 2


用于实现 $y = e^x$ 计算的类

In [ ]:
import numpy as np

class Exp(Function):
    def forward(self, x):
        return np.exp(x);

### 函数的连续调用，实现 $ y = ({e^x}^2)^2 $

In [ ]:
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.5))
a = A(x)
b = B(a)
y = C(b)

print(y.data)


数值微分实现, 或者导数的实现 $$f'(x) = \lim_{h \to 0} \frac{f(x+h)-f(x)}{h}$$

In [ ]:
def numerical_diff(f, x, eps=1e-4):
    x0 = Variable(x.datta - eps)
    x1 = Variable(x.data + eps)
    y0 = f(x0)
    y1 = f(x1)
    return (y1.data - y0.data) / (2*eps)


复合函数的导数的实现, 以  $ y = ({e^x}^2)^2 $ 为例

In [ ]:
def f(x):
    A = Square()
    B = Exp()
    C = Square()
    return C(B(A(x)))

x = Variable(np.array(0.5))
dy = numerical_diff(f, x)
print(dy.data)

### 反向传播
 Variable类的扩展

In [1]:
import numpy as np
class Variable:
    def __init__(self, data):
        self.data = data
        self.grad = None    # 导数值


Function类的扩展

In [4]:
class Function:
    def __call__(self, input):
        x = input.data
        # y = x ** 2
        y = self.forward(x) # 集体计算方法在forward中进行
        output = Variable(y)
        self.input = input  # 保存输入的变量
        return output

    def forward(self, x):
        raise NotImplementedError()
    # 计算导数的反向传播功能
    def backward(self, gy):
        raise NotImplementedError()

Square 和 Exp 功能的扩展

In [5]:
class Square(Function):
    def forward(self, x):
        return x ** 2

    def backward(self, gy):
        x = self.input.data
        gx = 2 * x * gy
        return gx


class Exp(Function):
    def forward(self, x):
        return np.exp(x);


    def backward(self, gy):
        x = self.input.data;
        gx = np.exp(x) * gy
        return gx


反向传播的实现

In [6]:
### 正向传播，实现 $ y = ({e^x}^2)^2 $
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.5))
a = A(x)
b = B(a)
y = C(b)

print(y.data)

y.grad = np.array(1.0)
b.grad = C.backward(y.grad)
a.grad = B.backward(b.grad)
x.grad = A.backward(a.grad)
print(x.grad)       # y 对 x 的导数值

1.648721270700128
3.297442541400256


反向传播的自动化

In [1]:
import numpy as np
class Variable:
    def __init__(self, data):
        self.data = data
        self.grad = None    # 导数值
        self.creator = None

    def set_creator(self, func):
        self.creator = func

class Function:
    def __call__(self, input):
        x = input.data
        # y = x ** 2
        y = self.forward(x) # 集体计算方法在forward中进行
        output = Variable(y)
        output.creator = self   # 让输出变量保存创造者信息

        self.input = input  # 保存输入的变量
        self.output = output    # 也保存输出变量
        return output

    def forward(self, x):
        raise NotImplementedError()
    # 计算导数的反向传播功能
    def backward(self, gy):
        raise NotImplementedError()

#region 函数实现
class Square(Function):
    def forward(self, x):
        return x ** 2

    def backward(self, gy):
        x = self.input.data
        gx = 2 * x * gy
        return gx


class Exp(Function):
    def forward(self, x):
        return np.exp(x);


    def backward(self, gy):
        x = self.input.data;
        gx = np.exp(x) * gy
        return gx
#endregion

### 正向传播，实现 $ y = ({e^x}^2)^2 $
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.5))
a = A(x)
b = B(a)
y = C(b)

print(y.data)

assert y.creator == C
assert y.creator.input == b
assert y.creator.input.creator == B
assert y.creator.input.creator.input == a
assert y.creator.input.creator.input.creator == A
assert y.creator.input.creator.input.creator.input == x


1.648721270700128
